# GPU：复用 CPU 缓存训练
选择 L4/A100 GPU 运行时，依次执行。先完成 CPU notebook，再运行这里。
原始数据路径、数据版本、滤波和采样参数必须与预热阶段一致。
`--require-cache` 禁止补算原始信号；缺失缓存会立即报错。
`--resume auto` 恢复同一目录的 last.ckpt；总轮数仍是 EPOCHS，不会额外再加 EPOCHS。


In [1]:
import os
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/wdongdong0708-jpg/brain2qwertyv1Test.git"
CODE = Path("/content/code")
WORKSPACE = Path("/content/drive/MyDrive/Brain2Qwerty_Workspace")

if (CODE / ".git").exists():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
os.chdir(CODE)

for folder in ("data", "cache_persisted", "archives", "runs"):
    (WORKSPACE / folder).mkdir(parents=True, exist_ok=True)

# 兼容 SpanishBCBL 和 SpanishBCBL_Full 两种命名
raw = WORKSPACE / "data/SpanishBCBL"
if not raw.is_dir():
    raw = WORKSPACE / "data/SpanishBCBL_Full"
if not raw.is_dir():
    raise FileNotFoundError(f"未找到数据集目录: {raw}")

# 与 CPU 预热阶段严格保持一致：在本地盘 /content/studies_links 建立软链接桥，确保 exca 缓存哈希 100% 命中！
LOCAL_STUDIES = Path("/content/studies_links")
LOCAL_STUDIES.mkdir(parents=True, exist_ok=True)
for name in ("Pinet2024Eeg", "Pinet2024Meg"):
    alias = LOCAL_STUDIES / name
    if alias.is_symlink() or alias.exists():
        alias.unlink()
    os.symlink(raw, alias)

os.environ["BRAIN2QWERTY_STUDIES"] = str(LOCAL_STUDIES)
os.environ["BRAIN2QWERTY_RESULTS"] = str(WORKSPACE / "runs")
!git log -1 --oneline
print("✅ 代码与数据桥接就绪！STUDIES ->", os.environ["BRAIN2QWERTY_STUDIES"])

Mounted at /content/drive
b0ddf75 (HEAD -> main, origin/main, origin/HEAD) Add unified local smoke and Colab cache/train workflow
✅ 代码与数据桥接就绪！STUDIES -> /content/studies_links


In [ ]:
%cd /content/code

# 1. 删掉刚才半途而废的残缺环境
!rm -rf /content/b2q-venv

# 2. 直接用 virtualenv 建好带 pip 且继承系统 PyTorch 的环境，
#    并把 colab_setup.py 里的 with_pip=True 改成 with_pip=False 防止它调 ensurepip 崩溃
!pip install -q virtualenv
!virtualenv --system-site-packages /content/b2q-venv
!sed -i 's/with_pip=True/with_pip=False/g' /content/code/scripts/colab_setup.py

# 3. 正式运行 colab_setup（这次它会自动生成 runtime-constraints.txt 并装好所有包！）
!python3 -m scripts.colab_setup

PYTHON = "/content/b2q-venv/bin/python"
!{PYTHON} -c "import torch, brain2qwerty_v1; print('✅ 环境完美就绪！GPU 可用:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')"

In [4]:
import os
os.environ["BRAIN2QWERTY_CACHE"] = "/content/local_cache"
ARCHIVE_FILE = WORKSPACE / "archives/features.tar"

!{PYTHON} -u -m scripts.cache_archive unpack \
    --archive "{ARCHIVE_FILE}" \
    --cache-root "{os.environ['BRAIN2QWERTY_CACHE']}"

Cache extracted to /content/local_cache


In [ ]:
MODALITY = "eeg"   # eeg / meg
SUBJECT = "all"    # 填 "002" 可先跑单受试者快速测试；填 "all" 跑全量受试者
EPOCHS = 300       # 总训练轮数（支持断点续训）
SEED = 33

# 使用 ! 直接调起，实时显示每个 Epoch 的进度条、Loss 与 CER 指标！
!{PYTHON} -u -m brain2qwerty_v1 train \
    --modality {MODALITY} \
    --subject {SUBJECT} \
    --seed {SEED} \
    --epochs {EPOCHS} \
    --devices 1 \
    --require-cache \
    --resume auto

Log-raw.events problem with: 003_DECOMEG_S1_9337_task1.vhdr
Log-raw.events problem with: 003_DECOMEG_S1_9337_task2.vhdr
Log-raw.events problem with: 004_DECOMEG_S2_NOID_task1.vhdr
Log-raw.events problem with: 004_DECOMEG_S2_noid_task2.vhdr
Log-raw.events problem with: 005_DECOMEG_S2_NOID_task1.vhdr
Log-raw.events problem with: 008_DECOMEG_S1_9846_task1.vhdr
Log-raw.events problem with: 008_DECOMEG_S1_9846_task2.vhdr
Log-raw.events problem with: 008_DECOMEG_S2_9846_task1.vhdr
Log-raw.events problem with: 008_DECOMEG_S2_9846_task2.vhdr
Log-raw.events problem with: 009_DECOMEG_S1_9949.vhdr
Log-raw.events problem with: 012_DECOMEG_S1_11481_task1.vhdr
Log-raw.events problem with: 013_DECOMEG_S1_11478_task1.vhdr
Missing log file for: 017_DECOMEG_S1_11743_task2.vhdr.
Missing log file for: 018_DECOMEG_S2_11662_task1.vhdr.
Missing log file for: 020_DECOMEG_S1_11966_task1.vhdr.
Missing log file for: 020_DECOMEG_S1_11966_task2.vhdr.
Log-raw.events problem with: 022_DECOMEG_S2_9948_task1.vhdr
Log-

Checkpoint、逐步日志、每轮状态均直接写入 Drive 的 runs/。同一实验再次执行会恢复断点；模型结构、batch、种子和总轮数需保持一致。新实验请指定不同的 `--run-dir`。
